# 📋 CV Matcher Pro — Notebook 02: Data Preprocessing (FIXED)

**Purpose**: Clean, normalize, and structure all 4 Kaggle datasets into
a unified format ready for triplet generation (Notebook 03).

**Key Fix**: Auto-discovers all CSV files in data/raw/ and maps columns
dynamically instead of hardcoding filenames.

**Outputs** (saved to `/content/drive/MyDrive/CVPRO/data/processed/`):
- `jd_clean.csv` — cleaned job descriptions with domain labels
- `resume_clean.csv` — chunked resume sections with domain labels
- `skills_by_domain.json` — top skills per domain (for backend enrichment)

**Runtime**: ~10-15 min on Colab

---
## Section 1: Setup & Configuration

In [1]:
# ============================================================================
# 1.1 — Mount Google Drive
# ============================================================================
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/CVPRO'
else:
    import os
    BASE_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))

print(f"BASE_DIR: {BASE_DIR}")

Mounted at /content/drive
BASE_DIR: /content/drive/MyDrive/CVPRO


In [2]:
# ============================================================================
# 1.2 — Import Libraries
# ============================================================================
import os
import re
import json
import hashlib
import warnings
import glob
import pandas as pd
import numpy as np
from collections import Counter

warnings.filterwarnings('ignore')
print("✅ Libraries imported")

✅ Libraries imported


In [3]:
# ============================================================================
# 1.3 — Define Paths
# ============================================================================
RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"📂 RAW_DIR:       {RAW_DIR}")
print(f"📂 PROCESSED_DIR: {PROCESSED_DIR}")

📂 RAW_DIR:       /content/drive/MyDrive/CVPRO/data/raw
📂 PROCESSED_DIR: /content/drive/MyDrive/CVPRO/data/processed


In [4]:
# ============================================================================
# 1.4 — Auto-discover ALL CSV files in raw directory
# ============================================================================
print("\n📂 All CSV files in data/raw/:")
print("=" * 70)
all_csvs = []
for root, dirs, files in os.walk(RAW_DIR):
    for f in sorted(files):
        if f.lower().endswith('.csv'):
            path = os.path.join(root, f)
            size_mb = os.path.getsize(path) / (1024*1024)
            # Quick peek at columns
            try:
                cols = list(pd.read_csv(path, nrows=0).columns)
            except:
                cols = ['ERROR reading']
            all_csvs.append({'path': path, 'filename': f, 'size_mb': size_mb, 'columns': cols})
            print(f"  📄 {f:45s} ({size_mb:8.2f} MB)  cols: {cols}")

print(f"\nTotal: {len(all_csvs)} CSV files found")


📂 All CSV files in data/raw/:
  📄 all_job_post.csv                              (    4.68 MB)  cols: ['job_id', 'category', 'job_title', 'job_description', 'job_skill_set']
  📄 job_skills.csv                                (  641.55 MB)  cols: ['job_link', 'job_skills']
  📄 job_summary.csv                               ( 4865.66 MB)  cols: ['job_link', 'job_summary']
  📄 job_title_des.csv                             (    4.38 MB)  cols: ['Unnamed: 0', 'Job Title', 'Job Description']
  📄 linkedin_job_postings.csv                     (  396.09 MB)  cols: ['job_link', 'last_processed_time', 'got_summary', 'got_ner', 'is_being_worked', 'job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type']
  📄 Resume.csv                                    (   53.67 MB)  cols: ['ID', 'Resume_str', 'Resume_html', 'Category']

Total: 6 CSV files found


---
## Section 2: Text Cleaning Functions

In [5]:
# ============================================================================
# 2.1 — Text Cleaning Functions
# ============================================================================
def clean_jd_text(text):
    """Clean a raw job description string."""
    if not isinstance(text, str) or not text.strip():
        return ''
    # Remove HTML tags
    text = re.sub(r'<[^>]*>', ' ', text)
    # Remove HTML entities
    text = re.sub(r'&[a-zA-Z]+;|&#\d+;', ' ', text)
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    # Remove emails
    text = re.sub(r'\S+@\S+\.\S+', ' ', text)
    # Remove phone numbers
    text = re.sub(r'\+?\d[\d\s\-\(\)]{8,15}\d', ' ', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def truncate_text(text, max_chars=2000):
    """Truncate text to ~512 tokens (rough proxy: 2000 chars)."""
    if len(text) <= max_chars:
        return text
    truncated = text[:max_chars]
    last_period = max(truncated.rfind('.'), truncated.rfind('\n'))
    if last_period > max_chars * 0.7:
        return truncated[:last_period + 1].strip()
    return truncated.strip()

def text_hash(text):
    """Generate hash for deduplication."""
    normalized = re.sub(r'\s+', ' ', text.lower().strip())[:1000]
    return hashlib.md5(normalized.encode('utf-8')).hexdigest()

print("✅ Text cleaning functions defined")

✅ Text cleaning functions defined


---
## Section 3: Smart Dataset Loading & Processing

Kita load setiap dataset berdasarkan **kolom yang ada**, bukan nama file.
Strategy:
- Cari CSV yang punya kolom `job_description` → JD data
- Cari CSV yang punya kolom `resume_str` → Resume data
- Cari CSV yang punya kolom `Job Description` → JD data (DS4)
- Cari CSV yang punya kolom `job_skills` → Skill extraction only

In [6]:
# ============================================================================
# 3.1 — Categorize each CSV by its content type
# ============================================================================
jd_files = []      # Files containing job description text
resume_files = []  # Files containing resume text
skill_files = []   # Files containing skill lists only
other_files = []   # Files we don't know how to use

for info in all_csvs:
    cols_lower = [c.lower() for c in info['columns']]

    if 'resume_str' in cols_lower or 'resume' in cols_lower:
        resume_files.append(info)
        print(f"  📝 RESUME:  {info['filename']}")
    elif any(c in cols_lower for c in ['job_description', 'description', 'job_desc',
                                         'job_summary', 'posting_text', 'job_details']):
        jd_files.append(info)
        print(f"  💼 JD TEXT: {info['filename']} → col: {[c for c in info['columns'] if c.lower() in ['job_description','description','job_desc','job_summary','posting_text','job_details']]}")
    elif 'job description' in cols_lower:
        jd_files.append(info)
        print(f"  💼 JD TEXT: {info['filename']} → col: Job Description")
    elif any(c in cols_lower for c in ['job_skills', 'skills']):
        skill_files.append(info)
        print(f"  🔧 SKILLS: {info['filename']}")
    else:
        other_files.append(info)
        print(f"  ❓ OTHER:  {info['filename']} → {info['columns'][:5]}")

print(f"\n📊 Summary: {len(jd_files)} JD files, {len(resume_files)} resume files, {len(skill_files)} skill files, {len(other_files)} other")

  💼 JD TEXT: all_job_post.csv → col: ['job_description']
  🔧 SKILLS: job_skills.csv
  💼 JD TEXT: job_summary.csv → col: ['job_summary']
  💼 JD TEXT: job_title_des.csv → col: Job Description
  ❓ OTHER:  linkedin_job_postings.csv → ['job_link', 'last_processed_time', 'got_summary', 'got_ner', 'is_being_worked']
  📝 RESUME:  Resume.csv

📊 Summary: 3 JD files, 1 resume files, 1 skill files, 1 other


---
## Section 4: Process JD Files

In [7]:
# ============================================================================
# 4.1 — Process all JD files
# ============================================================================
jd_frames = []

for info in jd_files:
    print(f"\n📌 Processing JD file: {info['filename']} ({info['size_mb']:.1f} MB)")
    print(f"   Columns: {info['columns']}")

    # Load (with nrows cap for very large files)
    nrows = None if info['size_mb'] < 100 else 500_000
    df = pd.read_csv(info['path'], low_memory=False, nrows=nrows)
    print(f"   Loaded: {len(df):,} rows")

    # Find JD text column
    jd_col = None
    for candidate in ['job_description', 'Job Description', 'description', 'Description',
                       'job_desc', 'job_summary', 'posting_text', 'job_details']:
        if candidate in df.columns:
            jd_col = candidate
            break
    # Fuzzy match fallback
    if jd_col is None:
        for col in df.columns:
            if 'desc' in col.lower() or 'summary' in col.lower():
                jd_col = col
                break

    if jd_col is None:
        print(f"   ⚠️ No JD text column found! Skipping.")
        continue

    print(f"   Using JD column: '{jd_col}'")

    # Find category column
    cat_col = None
    for candidate in ['category', 'Category', 'job_category', 'industry', 'Industry',
                       'sector', 'Sector', 'job_title', 'Job Title', 'title', 'Title',
                       'department', 'job_function']:
        if candidate in df.columns:
            cat_col = candidate
            break

    # Build clean dataframe
    result = pd.DataFrame()
    result['jd_text'] = df[jd_col].apply(clean_jd_text)
    result['raw_category'] = df[cat_col].astype(str) if cat_col else 'Unknown'
    result['source'] = info['filename'].replace('.csv', '')

    # Filter by length
    result['char_len'] = result['jd_text'].str.len()
    before = len(result)
    result = result[(result['char_len'] >= 100) & (result['char_len'] <= 5000)].copy()
    print(f"   Length filter: {before:,} → {len(result):,} (removed {before-len(result):,})")

    # Deduplicate
    result['text_hash'] = result['jd_text'].apply(text_hash)
    before = len(result)
    result = result.drop_duplicates(subset='text_hash').copy()
    print(f"   Dedup: {before:,} → {len(result):,}")

    # Truncate
    result['jd_text'] = result['jd_text'].apply(truncate_text)

    jd_frames.append(result[['jd_text', 'raw_category', 'source']])
    print(f"   ✅ Processed: {len(result):,} JDs")


📌 Processing JD file: all_job_post.csv (4.7 MB)
   Columns: ['job_id', 'category', 'job_title', 'job_description', 'job_skill_set']
   Loaded: 1,167 rows
   Using JD column: 'job_description'
   Length filter: 1,167 → 887 (removed 280)
   Dedup: 887 → 869
   ✅ Processed: 869 JDs

📌 Processing JD file: job_summary.csv (4865.7 MB)
   Columns: ['job_link', 'job_summary']
   Loaded: 500,000 rows
   Using JD column: 'job_summary'
   Length filter: 500,000 → 391,300 (removed 108,700)
   Dedup: 391,300 → 282,192
   ✅ Processed: 282,192 JDs

📌 Processing JD file: job_title_des.csv (4.4 MB)
   Columns: ['Unnamed: 0', 'Job Title', 'Job Description']
   Loaded: 2,277 rows
   Using JD column: 'Job Description'
   Length filter: 2,277 → 2,189 (removed 88)
   Dedup: 2,189 → 2,157
   ✅ Processed: 2,157 JDs


In [8]:
# ============================================================================
# 4.2 — Combine all JD data
# ============================================================================
if jd_frames:
    jd_all = pd.concat(jd_frames, ignore_index=True)

    # Global dedup
    jd_all['text_hash'] = jd_all['jd_text'].apply(text_hash)
    before = len(jd_all)
    jd_all = jd_all.drop_duplicates(subset='text_hash').drop(columns='text_hash')
    print(f"\n📊 Combined JD Dataset: {before:,} → {len(jd_all):,}")
    print(f"   Per source:\n{jd_all['source'].value_counts().to_string()}")
else:
    jd_all = pd.DataFrame(columns=['jd_text', 'raw_category', 'source'])
    print("⚠️ No JD data found! Check your data/raw/ directory.")


📊 Combined JD Dataset: 285,218 → 285,206
   Per source:
source
job_summary      282180
job_title_des      2157
all_job_post        869


---
## Section 5: Process Resume Data (Dataset 3)

In [9]:
# ============================================================================
# 5.1 — Process Resume files
# ============================================================================
resume_frames = []

for info in resume_files:
    print(f"\n📌 Processing Resume file: {info['filename']} ({info['size_mb']:.1f} MB)")
    print(f"   Columns: {info['columns']}")

    df = pd.read_csv(info['path'])
    print(f"   Loaded: {len(df):,} rows")

    # Find text column
    text_col = None
    for candidate in ['resume_str', 'Resume_str', 'resume', 'Resume', 'text', 'content']:
        if candidate in df.columns:
            text_col = candidate
            break
    if text_col is None:
        # Fallback: pick longest text column
        for col in df.columns:
            if df[col].dtype == 'object':
                avg_len = df[col].astype(str).str.len().mean()
                if avg_len > 200:
                    text_col = col
                    break

    if text_col is None:
        print(f"   ⚠️ No resume text column found! Skipping.")
        continue

    # Find category column
    cat_col = None
    for candidate in ['Category', 'category', 'Label', 'label', 'class', 'job_category']:
        if candidate in df.columns:
            cat_col = candidate
            break

    print(f"   Using text column: '{text_col}', category column: '{cat_col}'")

    # Process each resume
    for _, row in df.iterrows():
        raw_text = str(row[text_col])
        category = str(row[cat_col]) if cat_col else 'Unknown'

        if len(raw_text) < 100:
            continue

        # Clean the resume text
        cleaned = clean_jd_text(raw_text)

        # Chunk resume into sections
        # Try splitting by common resume headers
        section_patterns = [
            r'(?i)\b(?:experience|work\s*experience|employment|professional\s*experience)\b',
            r'(?i)\b(?:education|academic|qualification|degree)\b',
            r'(?i)\b(?:skills|technical\s*skills|competenc|proficienc)\b',
            r'(?i)\b(?:summary|objective|profile|about)\b',
            r'(?i)\b(?:certif|licens|training|course)\b',
            r'(?i)\b(?:project|portfolio|achievement|award)\b',
        ]

        # Split by headers — create chunks
        chunks = []
        # Also add the full resume as a chunk (for overall matching)
        full_truncated = truncate_text(cleaned, 2000)
        if len(full_truncated) >= 100:
            chunks.append(full_truncated)

        # Split by double newlines or major headers
        sections = re.split(r'\n\s*\n|(?=(?:EXPERIENCE|EDUCATION|SKILLS|SUMMARY|PROJECTS|CERTIF))',
                           cleaned, flags=re.IGNORECASE)
        for section in sections:
            section = section.strip()
            if len(section) >= 100:
                chunks.append(truncate_text(section, 2000))

        # If no good sections, just use the full text
        if not chunks:
            if len(cleaned) >= 50:
                chunks.append(truncate_text(cleaned, 2000))

        for chunk in chunks:
            resume_frames.append({
                'resume_chunk': chunk,
                'full_category': category,
                'source': info['filename'].replace('.csv', '')
            })

    print(f"   ✅ Processed: {len(df):,} resumes → {len(resume_frames):,} chunks so far")

resume_all = pd.DataFrame(resume_frames) if resume_frames else pd.DataFrame(columns=['resume_chunk', 'full_category', 'source'])
print(f"\n📊 Total Resume Chunks: {len(resume_all):,}")
if len(resume_all) > 0:
    print(f"   Categories: {resume_all['full_category'].nunique()}")
    print(f"   Per category:\n{resume_all['full_category'].value_counts().head(15).to_string()}")


📌 Processing Resume file: Resume.csv (53.7 MB)
   Columns: ['ID', 'Resume_str', 'Resume_html', 'Category']
   Loaded: 2,484 rows
   Using text column: 'Resume_str', category column: 'Category'
   ✅ Processed: 2,484 resumes → 21,480 chunks so far

📊 Total Resume Chunks: 21,480
   Categories: 24
   Per category:
full_category
CONSTRUCTION              1211
ADVOCATE                  1134
ENGINEERING               1101
INFORMATION-TECHNOLOGY    1073
TEACHER                   1050
HEALTHCARE                1049
CONSULTANT                 998
FITNESS                    986
ARTS                       977
PUBLIC-RELATIONS           969
AVIATION                   962
BANKING                    950
DESIGNER                   949
ACCOUNTANT                 942
BUSINESS-DEVELOPMENT       939


---
## Section 6: Category Normalization → Domain Labels

**Strategy (2-pass)**:
1. Map dari raw_category/job_title (fast, exact)
2. Kalau masih "Other" → scan isi JD text untuk infer domain (keyword scoring)

In [10]:
# ============================================================================
# 6.1 — Domain Keyword Dictionaries (untuk scoring)
# ============================================================================
# Setiap domain punya list keyword. Semakin banyak keyword match di title+text,
# semakin yakin kita tentang domain-nya.

DOMAIN_KEYWORDS = {
    'IT': [
        'software', 'developer', 'engineer', 'programming', 'python', 'java',
        'javascript', 'react', 'angular', 'node.js', 'sql', 'database',
        'cloud', 'aws', 'azure', 'devops', 'docker', 'kubernetes',
        'machine learning', 'data science', 'artificial intelligence', 'ai ',
        'backend', 'frontend', 'fullstack', 'full-stack', 'full stack',
        'web develop', 'mobile develop', 'ios', 'android', 'api',
        'cybersecurity', 'cyber security', 'information security',
        'network engineer', 'system admin', 'sysadmin', 'linux', 'unix',
        'agile', 'scrum', 'git', 'ci/cd', 'microservice',
        'data engineer', 'data analyst', 'etl', 'hadoop', 'spark',
        'tensorflow', 'pytorch', 'deep learning', 'nlp', 'computer vision',
        'qa engineer', 'quality assurance', 'test automation', 'selenium',
        'it support', 'helpdesk', 'technical support', 'sre', 'reliability',
        'blockchain', 'smart contract', 'ruby', 'golang', 'rust', 'c++',
        'php', 'laravel', 'django', '.net', 'typescript', 'vue',
        'html', 'css', 'xml', 'json', 'rest api', 'graphql',
        'tableau', 'power bi', 'looker', 'dbt',
    ],
    'Finance': [
        'finance', 'financial', 'accounting', 'accountant', 'auditor', 'audit',
        'banking', 'bank ', 'investment', 'investor', 'portfolio',
        'tax ', 'taxation', 'revenue', 'treasury', 'cfo', 'controller',
        'bookkeep', 'payroll', 'accounts payable', 'accounts receivable',
        'credit analyst', 'risk analyst', 'underwriting', 'mortgage',
        'insurance', 'actuary', 'actuarial', 'financial planning',
        'budget', 'forecast', 'gaap', 'ifrs', 'sec filing',
        'equity', 'bond', 'derivative', 'asset management', 'wealth',
        'compliance officer', 'financial control', 'cost account',
    ],
    'Marketing': [
        'marketing', 'brand', 'branding', 'advertising', 'advertise',
        'social media', 'content creat', 'content market', 'content strat',
        'seo', 'sem', 'ppc', 'google ads', 'facebook ads',
        'email marketing', 'campaign', 'digital marketing',
        'copywriter', 'copywriting', 'creative direct', 'art direct',
        'graphic design', 'ui/ux', 'ux design', 'ui design', 'user experience',
        'product design', 'visual design', 'interaction design',
        'public relation', 'pr manager', 'communications',
        'market research', 'consumer insight', 'analytics',
        'growth hack', 'growth market', 'performance market',
        'influencer', 'affiliate', 'media buy', 'media plan',
        'video production', 'photography', 'illustration',
    ],
    'Sales': [
        'sales', 'selling', 'account executive', 'account manager',
        'business development', 'bdr', 'sdr', 'inside sales',
        'outside sales', 'field sales', 'territory', 'quota',
        'lead generation', 'prospecting', 'cold call', 'pipeline',
        'crm', 'salesforce', 'hubspot', 'customer acquisition',
        'revenue growth', 'client relation', 'client management',
        'key account', 'enterprise sales', 'channel partner',
        'retail sales', 'store manager', 'commercial',
    ],
    'HR': [
        'human resource', 'hr manager', 'hr business partner', 'hrbp',
        'recruiter', 'recruiting', 'recruitment', 'talent acquisition',
        'talent management', 'people operation', 'people team',
        'employee relation', 'labor relation', 'workforce',
        'onboarding', 'offboarding', 'performance review',
        'compensation', 'benefits', 'total rewards',
        'learning and development', 'l&d', 'training and development',
        'diversity', 'inclusion', 'dei', 'organizational development',
        'hr generalist', 'hr specialist', 'hr coordinator',
        'staffing', 'headhunt', 'employer brand',
    ],
    'Healthcare': [
        'healthcare', 'health care', 'medical', 'clinical',
        'nurse', 'nursing', 'rn ', 'registered nurse', 'lpn',
        'physician', 'doctor', 'surgeon', 'md ',
        'pharmacy', 'pharmacist', 'pharmaceutical', 'pharma',
        'therapist', 'therapy', 'physical therapy', 'occupational therapy',
        'dental', 'dentist', 'orthodont', 'oral health',
        'hospital', 'patient', 'diagnosis', 'treatment plan',
        'laboratory', 'lab technician', 'patholog', 'radiolog',
        'mental health', 'psycholog', 'psychiatr', 'counselor',
        'emt', 'paramedic', 'ambulance', 'emergency medicine',
        'biotech', 'biotechnology', 'life science', 'genomic',
        'veterinar', 'optometr', 'chiropract', 'midwi',
        'fitness', 'wellness', 'nutrition', 'dietitian',
    ],
    'Education': [
        'teacher', 'teaching', 'professor', 'lecturer', 'instructor',
        'curriculum', 'pedagogy', 'classroom', 'school',
        'education', 'academic', 'university', 'college',
        'tutor', 'tutoring', 'e-learning', 'elearning',
        'student', 'dean', 'principal', 'superintendent',
        'special education', 'sped', 'early childhood',
        'training specialist', 'instructional design',
        'librarian', 'research assistant', 'postdoc',
    ],
    'Legal': [
        'legal', 'lawyer', 'attorney', 'law firm', 'law clerk',
        'paralegal', 'litigation', 'corporate counsel', 'general counsel',
        'compliance', 'regulatory', 'contract', 'intellectual property',
        'patent', 'trademark', 'copyright', 'mergers and acquisition',
        'arbitration', 'mediation', 'court', 'judge',
        'advocate', 'solicitor', 'barrister', 'notary',
        'privacy', 'gdpr', 'data protection', 'hipaa',
    ],
    'Operational': [
        'operations', 'logistics', 'supply chain', 'warehouse',
        'procurement', 'purchasing', 'vendor management',
        'manufacturing', 'production', 'quality control', 'lean',
        'six sigma', 'process improvement', 'continuous improvement',
        'inventory', 'distribution', 'shipping', 'freight',
        'facilities', 'maintenance', 'safety', 'osha',
        'construction', 'architecture', 'civil engineer',
        'automotive', 'mechanic', 'technician',
        'aviation', 'pilot', 'flight', 'airline',
        'hospitality', 'hotel', 'restaurant', 'food service', 'chef',
        'agriculture', 'farming', 'agri',
        'real estate', 'property manage',
        'customer service', 'call center', 'support specialist',
    ],
}

def infer_domain_from_text(title, jd_text=''):
    """
    Infer domain by scoring keyword matches in title + JD text.
    Title matches are weighted 3x higher than text body matches.
    Returns (domain, confidence_score).
    """
    title_lower = str(title).lower()
    text_lower = str(jd_text).lower()[:3000]  # cap for speed
    combined = title_lower + ' ' + text_lower

    scores = {}
    for domain, keywords in DOMAIN_KEYWORDS.items():
        score = 0
        for kw in keywords:
            # Title match = 3 points
            if kw in title_lower:
                score += 3
            # Text body match = 1 point
            if kw in text_lower:
                score += 1
        if score > 0:
            scores[domain] = score

    if not scores:
        return 'Other', 0

    best_domain = max(scores, key=scores.get)
    best_score = scores[best_domain]

    # Need minimum confidence (at least 2 points)
    if best_score < 2:
        return 'Other', best_score

    return best_domain, best_score


print("✅ Domain keyword dictionaries defined")
print(f"   Total domains: {len(DOMAIN_KEYWORDS)}")
print(f"   Total keywords: {sum(len(v) for v in DOMAIN_KEYWORDS.values())}")

✅ Domain keyword dictionaries defined
   Total domains: 9
   Total keywords: 391


In [11]:
# ============================================================================
# 6.2 — Apply Domain Mapping to JD Data (2-pass)
# ============================================================================
if len(jd_all) > 0:
    print("📊 Applying domain mapping to JD data...")
    print("   Pass 1: Map from raw_category (job title)...")

    # Pass 1: Quick category/title mapping
    domains = []
    confidences = []
    for _, row in jd_all.iterrows():
        domain, conf = infer_domain_from_text(row['raw_category'], row['jd_text'])
        domains.append(domain)
        confidences.append(conf)

    jd_all['domain'] = domains
    jd_all['domain_confidence'] = confidences

    n_other = (jd_all['domain'] == 'Other').sum()
    print(f"   After mapping: {len(jd_all) - n_other:,} classified, {n_other:,} still 'Other'")

    print("\n📊 JD Domain Distribution:")
    print(jd_all['domain'].value_counts().to_string())

    # Show confidence stats
    print(f"\n📊 Domain Confidence Stats:")
    print(f"   Mean: {jd_all['domain_confidence'].mean():.1f}")
    print(f"   Median: {jd_all['domain_confidence'].median():.1f}")

# Apply to Resume data (simpler — categories are clean)
if len(resume_all) > 0:
    domains_r = []
    for _, row in resume_all.iterrows():
        domain, _ = infer_domain_from_text(row['full_category'], row.get('resume_chunk', ''))
        domains_r.append(domain)
    resume_all['domain'] = domains_r
    print("\n📊 Resume Domain Distribution:")
    print(resume_all['domain'].value_counts().to_string())

📊 Applying domain mapping to JD data...
   Pass 1: Map from raw_category (job title)...
   After mapping: 261,113 classified, 24,093 still 'Other'

📊 JD Domain Distribution:
domain
Healthcare     97932
Operational    54281
IT             29188
Other          24093
Finance        22552
HR             15017
Education      14899
Sales          12019
Legal           9376
Marketing       5849

📊 Domain Confidence Stats:
   Mean: 4.3
   Median: 4.0

📊 Resume Domain Distribution:
domain
Operational    4374
Finance        3428
IT             2602
Other          2589
Education      2540
Healthcare     2239
Sales          1252
Marketing       999
Legal           937
HR              520


---
## Section 7: Extract Skills from Skill Files

In [12]:
# ============================================================================
# 7.1 — Extract skills per domain from skill files
# ============================================================================
skills_by_domain = {}

# Process dedicated skill files
for info in skill_files:
    print(f"\n🔧 Processing skills from: {info['filename']}")
    df = pd.read_csv(info['path'], low_memory=False)
    print(f"   Columns: {list(df.columns)}")

    # Find skills column
    skill_col = None
    for candidate in ['job_skills', 'job_skill_set', 'Skills', 'skills', 'skill_set']:
        if candidate in df.columns:
            skill_col = candidate
            break
    if skill_col is None:
        for col in df.columns:
            if 'skill' in col.lower():
                skill_col = col
                break

    if skill_col is None:
        print(f"   ⚠️ No skills column found, skipping")
        continue

    print(f"   Using skills column: '{skill_col}'")

    # Extract individual skills
    all_skills = []
    for val in df[skill_col].dropna():
        val = str(val)
        # Skills might be comma-separated, semicolon-separated, or in list format
        if val.startswith('[') or val.startswith("'"):
            # Python list-like format
            val = re.sub(r"[\[\]']", '', val)
        skills = re.split(r'[,;|]', val)
        for s in skills:
            s = s.strip().strip("'\"")
            if len(s) >= 2 and len(s) <= 50:
                all_skills.append(s)

    skill_counts = Counter(all_skills)
    top_skills = [s for s, _ in skill_counts.most_common(100)]
    skills_by_domain['_all_from_skill_files'] = top_skills
    print(f"   Extracted {len(skill_counts):,} unique skills (top 10: {top_skills[:10]})")

# Also extract skills from JD text (keyword-based from domain configs)
# This enriches the skill vocabulary per domain
if len(jd_all) > 0:
    for domain in jd_all['domain'].unique():
        domain_jds = jd_all[jd_all['domain'] == domain]['jd_text'].tolist()
        # Simple word extraction from JD titles/categories
        domain_cats = jd_all[jd_all['domain'] == domain]['raw_category'].value_counts()
        skills_by_domain[domain] = {
            'n_jds': len(domain_jds),
            'top_categories': domain_cats.head(10).to_dict()
        }

print(f"\n📊 Skills data collected for {len(skills_by_domain)} groups")


🔧 Processing skills from: job_skills.csv
   Columns: ['job_link', 'job_skills']
   Using skills column: 'job_skills'
   Extracted 3,006,029 unique skills (top 10: ['Communication', 'Teamwork', 'Leadership', 'Customer service', 'Communication skills', 'Customer Service', 'Problem Solving', 'Sales', 'Problemsolving', 'Nursing'])

📊 Skills data collected for 11 groups


---
## Section 8: Save Processed Data

In [13]:
# ============================================================================
# 8.1 — Save JD data
# ============================================================================
jd_save_path = os.path.join(PROCESSED_DIR, 'jd_clean.csv')
if len(jd_all) > 0:
    jd_save = jd_all[['jd_text', 'raw_category', 'domain', 'source']].copy()
    jd_save.to_csv(jd_save_path, index=False)
    print(f"✅ Saved JD data: {jd_save_path}")
    print(f"   Rows: {len(jd_save):,}")
    print(f"   Domains: {jd_save['domain'].nunique()}")
    print(f"   Sources: {jd_save['source'].nunique()}")
else:
    print("⚠️ No JD data to save!")

✅ Saved JD data: /content/drive/MyDrive/CVPRO/data/processed/jd_clean.csv
   Rows: 285,206
   Domains: 10
   Sources: 3


In [14]:
# ============================================================================
# 8.2 — Save Resume data
# ============================================================================
resume_save_path = os.path.join(PROCESSED_DIR, 'resume_clean.csv')
if len(resume_all) > 0:
    resume_save = resume_all[['resume_chunk', 'full_category', 'domain', 'source']].copy()
    resume_save.to_csv(resume_save_path, index=False)
    print(f"✅ Saved Resume data: {resume_save_path}")
    print(f"   Rows: {len(resume_save):,}")
    print(f"   Domains: {resume_save['domain'].nunique()}")
    print(f"   Categories: {resume_save['full_category'].nunique()}")
else:
    print("⚠️ No Resume data to save!")

✅ Saved Resume data: /content/drive/MyDrive/CVPRO/data/processed/resume_clean.csv
   Rows: 21,480
   Domains: 10
   Categories: 24


In [15]:
# ============================================================================
# 8.3 — Save Skills data
# ============================================================================
skills_save_path = os.path.join(PROCESSED_DIR, 'skills_by_domain.json')
# Convert any non-serializable data
skills_output = {}
for key, val in skills_by_domain.items():
    if isinstance(val, list):
        skills_output[key] = val
    elif isinstance(val, dict):
        skills_output[key] = {str(k): v for k, v in val.items()}
    else:
        skills_output[key] = str(val)

with open(skills_save_path, 'w', encoding='utf-8') as f:
    json.dump(skills_output, f, indent=2, ensure_ascii=False)
print(f"✅ Saved Skills data: {skills_save_path}")

✅ Saved Skills data: /content/drive/MyDrive/CVPRO/data/processed/skills_by_domain.json


In [16]:
# ============================================================================
# 8.4 — Final Summary
# ============================================================================
print("\n" + "=" * 60)
print("📊 PREPROCESSING COMPLETE — SUMMARY")
print("=" * 60)

if len(jd_all) > 0:
    print(f"\n💼 JD Data:")
    print(f"   Total rows:     {len(jd_all):,}")
    print(f"   Avg text length: {jd_all['jd_text'].str.len().mean():.0f} chars")
    print(f"   Domains:        {jd_all['domain'].nunique()}")
    print(f"   Domain breakdown:")
    for domain, count in jd_all['domain'].value_counts().items():
        print(f"     {domain:15s}: {count:,}")

if len(resume_all) > 0:
    print(f"\n📝 Resume Data:")
    print(f"   Total chunks:    {len(resume_all):,}")
    print(f"   Avg chunk length: {resume_all['resume_chunk'].str.len().mean():.0f} chars")
    print(f"   Domains:         {resume_all['domain'].nunique()}")
    print(f"   Domain breakdown:")
    for domain, count in resume_all['domain'].value_counts().items():
        print(f"     {domain:15s}: {count:,}")

print(f"\n📊 Overlapping Domains (JD ∩ Resume):")
if len(jd_all) > 0 and len(resume_all) > 0:
    jd_domains = set(jd_all['domain'].unique())
    resume_domains = set(resume_all['domain'].unique())
    overlap = jd_domains & resume_domains
    for d in sorted(overlap):
        jd_n = len(jd_all[jd_all['domain'] == d])
        res_n = len(resume_all[resume_all['domain'] == d])
        print(f"   ✅ {d:15s}: {jd_n:,} JDs + {res_n:,} resume chunks")
    jd_only = jd_domains - resume_domains
    if jd_only:
        print(f"   ⚠️ JD only (no resume data): {jd_only}")
    res_only = resume_domains - jd_domains
    if res_only:
        print(f"   ⚠️ Resume only (no JD data): {res_only}")

print(f"\n🚀 Next step: Run Notebook 03 (Triplet Generation)")
print("=" * 60)


📊 PREPROCESSING COMPLETE — SUMMARY

💼 JD Data:
   Total rows:     285,206
   Avg text length: 1693 chars
   Domains:        10
   Domain breakdown:
     Healthcare     : 97,932
     Operational    : 54,281
     IT             : 29,188
     Other          : 24,093
     Finance        : 22,552
     HR             : 15,017
     Education      : 14,899
     Sales          : 12,019
     Legal          : 9,376
     Marketing      : 5,849

📝 Resume Data:
   Total chunks:    21,480
   Avg chunk length: 758 chars
   Domains:         10
   Domain breakdown:
     Operational    : 4,374
     Finance        : 3,428
     IT             : 2,602
     Other          : 2,589
     Education      : 2,540
     Healthcare     : 2,239
     Sales          : 1,252
     Marketing      : 999
     Legal          : 937
     HR             : 520

📊 Overlapping Domains (JD ∩ Resume):
   ✅ Education      : 14,899 JDs + 2,540 resume chunks
   ✅ Finance        : 22,552 JDs + 3,428 resume chunks
   ✅ HR             : 1